# Steam Dataset Cleaning

In [ ]:
import ast
import csv
import json
import os
import re
import shutil
import warnings
from datetime import datetime
from pathlib import Path

import joblib
from joblib import Memory
import lime
import lime.lime_tabular
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
import torch
import optuna
from optuna.distributions import CategoricalDistribution, FloatDistribution, IntDistribution
from optuna.integration import OptunaSearchCV
from scipy.optimize import differential_evolution
from scipy.stats import friedmanchisquare, wilcoxon
from sentence_transformers import SentenceTransformer

import sklearn
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectFromModel
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, cohen_kappa_score, confusion_matrix, ConfusionMatrixDisplay, f1_score, make_scorer, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder, RobustScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight

import xgboost as xgb
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTENC
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler, TomekLinks
from tqdm.auto import tqdm

# Global Configuration
warnings.filterwarnings('ignore', category=UserWarning)
sklearn.set_config(transform_output="pandas")

SEED = 1
PRE_RELEASE = True
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
qwk_scorer = make_scorer(cohen_kappa_score, weights='quadratic')

In [ ]:
"""
Utility Functions
├─ parse_dict
├─ extract_list_from_dicts
├─ extract_keys_from_dict
└─ parse_simple_list
"""
def parse_dict(val):
    """
    Safely convert a stringified dictionary into a Python dict.
    Handles NaN, None, real dicts, or malformed strings.
    Returns parsed dict or {} on failure.
    Uses ast.literal_eval (safe alternative to eval).
    Designed to be defensive: any parsing error is absorbed
    to prevent pipeline failures in downstream steps.
    """
    if pd.isna(val): return {}
    try: return ast.literal_eval(val)
    except: return {}

def extract_list_from_dicts(val):
    """
    Convert Steam-style list-of-dicts into ML-ready token list.
    Extracts 'description' from each dict, normalizes text by replacing
    spaces with underscores, and ignores invalid entries.
    Accepts stringified lists, real lists, or NaN values.
    Returns cleaned list or [] if input is invalid/unparseable.
    Fail-safe: any parsing error is absorbed to keep pipeline stable.
    """
  
    if pd.isna(val): return []
    try:
        parsed = ast.literal_eval(val)
        if isinstance(parsed, list):
            return [str(item.get('description', '')).replace(' ', '_') for item in parsed if isinstance(item, dict)]
        return []
    except: return []

def extract_keys_from_dict(val, top_n=5):
    """
    Extract top-N highest scoring keys from a dict-like structure.
    Used mainly for SteamSpy tags where values represent relevance scores.
    Parses input safely via parse_dict() and returns empty list on failure.
    Output is sorted by value (descending) and normalized with underscores.
    Returns up to top_n keys as ML-ready tokens.
    Fail-safe: malformed or missing data never breaks the pipeline.
    """

    parsed = parse_dict(val)
    if not parsed: return []
    # Top n tags
    sorted_tags = sorted(parsed.items(), key=lambda item: item[1], reverse=True)[:top_n]
    return [str(tag[0]).replace(' ', '_') for tag in sorted_tags]

# Parsing string lists
def parse_simple_list(val):
    """
    Parse list-like values into a normalized list of strings.
    Supports stringified Python lists, comma-separated strings,
    and missing values.
    Normalizes items by trimming whitespace and replacing spaces
    with underscores.
    Falls back to comma splitting if literal_eval() fails.
    Returns cleaned list or [] for empty, invalid, or malformed input.
    """

    if pd.isna(val) or val == '[]': 
        return []
    try:
        parsed = ast.literal_eval(val)
        if isinstance(parsed, list):
            return [str(item).strip().replace(' ', '_') for item in parsed]
        return []
    except (ValueError, SyntaxError):
        if isinstance(val, str):
            return [item.strip().replace(' ', '_') for item in val.split(',') if item.strip()]
        return []

"""
Dataset Loading
└─ load_and_merge
"""
def load_and_merge(steam_app_path, steam_spy_path):
    """
    Load Steam Store and SteamSpy datasets and merge them into a
    single DataFrame.
    Joins both sources using the common Steam application identifier
    (steam_appid in Store, appid in SteamSpy).
    Applies '_store' and '_spy' suffixes to overlapping column names.
    Returns a unified dataset combining metadata, player statistics,
    tags, reviews, and ownership information for ML processing.
    """
    print("Loading and merging original datasets...")

    # Skipping bad lines
    try:
        df_store = pd.read_csv(steam_app_path, on_bad_lines='skip', engine='python')
        df_spy = pd.read_csv(steam_spy_path, on_bad_lines='skip', engine='python')
    except TypeError:
        df_store = pd.read_csv(steam_app_path, error_bad_lines=False, engine='python')
        df_spy = pd.read_csv(steam_spy_path, error_bad_lines=False, engine='python')

    # Merging
    df = pd.merge(df_store, df_spy, left_on='steam_appid', right_on='appid', suffixes=('_store', '_spy'))

    # Check on the schema
    # If a malformed line had exactly the same number of commas, but the data 
    # still shifted (e.g., the description text ended up in the price column), 
    # the ID column (which must be purely numeric) will contain text fragments.
    initial_len = len(df)
    
    # We force the ID to be numeric. Anything that's shifted text will become NaN
    df['steam_appid'] = pd.to_numeric(df['steam_appid'], errors='coerce')
    
    # Dropping shifted lines
    df = df.dropna(subset=['steam_appid']).copy()
    
    dropped = initial_len - len(df)
    if dropped > 0:
        print(f"   -> Dropped {dropped} malformed/shifted rows post-merge.")

    # Cast back the ID into int
    df['steam_appid'] = df['steam_appid'].astype(int)
    return df

In [ ]:
"""
Target Engineering
└─ process_target_owners
"""
def process_target_owners(df):
    """
    Convert SteamSpy ownership ranges into 5 ordinal target tiers.
    Mapping logic derived from project architecture:
    Tier 0: 0 .. 20k           (The Indie Long-Tail - Low adoption)
    Tier 1: 20k .. 100k        (Healthy Niche - Sustainable indie)
    Tier 2: 100k .. 500k       (Mid-Market Success - Breakout hits)
    Tier 3: 500k .. 2M         (Major Success - AA level)
    Tier 4: 2M+                (Hit / AAA status)
    """
    print("Processing the target: 'owners' (Re-Binning in 5 Tiers)...")
    
    steamspy_tiers = {
        # Tier 0
        '0 .. 20,000': 0, 
        
        # Tier 1
        '20,000 .. 50,000': 1, 
        '50,000 .. 100,000': 1,
        
        # Tier 2
        '100,000 .. 200,000': 2, 
        '200,000 .. 500,000': 2, 
        
        # Tier 3
        '500,000 .. 1,000,000': 3, 
        '1,000,000 .. 2,000,000': 3,
        
        # Tier 4
        '2,000,000 .. 5,000,000': 4, 
        '5,000,000 .. 10,000,000': 4,
        '10,000,000 .. 20,000,000': 4, 
        '20,000,000 .. 50,000,000': 4, 
        '50,000,000 .. 100,000,000': 4, 
        '100,000,000 .. 200,000,000': 4
    }
    
    # Map raw SteamSpy ranges to ordinal tiers
    df['target_owners'] = df['owners'].map(steamspy_tiers)
    
    # Check for any unmapped (invalid or missing) 'owners' values
    unmapped = df['target_owners'].isna().sum()
    if unmapped > 0:
        print(f"   -> WARNING: {unmapped} lines dropped (invalid or missing 'owners' range)")
    
    # Drop rows with a null target and cast to integer
    df = df.dropna(subset=['target_owners']).copy()
    df['target_owners'] = df['target_owners'].astype(int)
    
    return df

"""
Basic Game Feature Engineering
└─ process_game_features
"""
# Filtering games and cleaning game features
def process_game_features(df):
    """
    Filter non-game entries and engineer basic gameplay features.
    Keeps only rows where type == 'game', excluding DLCs, software,
    videos, and other store items.
    Creates:
    - is_controller_supported (binary flag)
    - num_dlc (DLC count)
    Converts raw metadata into simple ML-friendly features.
    """
    print("Cleaning type, controller support, and dlc features...")
    
    # Filtering on game
    if 'type' in df.columns:
        initial_len = len(df)
        df = df[df['type'] == 'game'].copy()
        print(f"   -> Dropped {initial_len - len(df)} non-game rows.")
        
    # Binarizing controller support
    if 'controller_support' in df.columns:
        df['is_controller_supported'] = df['controller_support'].notna().astype(int)
        df = df.drop(columns=['controller_support'])
        
    # Counting DLCs
    if 'dlc' in df.columns:
        def count_dlcs(val):
            if pd.isna(val) or val == '[]': 
                return 0
            if isinstance(val, str):
                try:
                    parsed = ast.literal_eval(val)
                    if isinstance(parsed, list): 
                        return len(parsed)
                except (ValueError, SyntaxError): 
                    return 0
            elif isinstance(val, list):
                return len(val)
            return 0
            
        df['num_dlc'] = df['dlc'].apply(count_dlcs)
        df = df.drop(columns=['dlc'])
        
    return df

"""
Restrictions Features
└─ extract_restrictions_features
"""
# Extract DRM and external account requirements
def extract_restrictions_features(df):
    """
    Convert DRM and external account requirements into binary features.
    Creates:
    - has_third_party_drm
    - requires_ext_account
    Uses the presence of restriction notices rather than their text,
    reducing feature complexity while preserving relevant signals.
    Original text columns are removed after feature extraction.
    """

    print("Extracting DRM and external account features...")
    
    if 'drm_notice' in df.columns:
        df['has_third_party_drm'] = df['drm_notice'].notna().astype(int)
        df = df.drop(columns=['drm_notice'])
        
    if 'ext_user_account_notice' in df.columns:
        df['requires_ext_account'] = df['ext_user_account_notice'].notna().astype(int)
        df = df.drop(columns=['ext_user_account_notice'])
        
    return df

"""
JSON Feature Extraction
├─ extract_achievements_total
└─ extract_json_features
"""
# Extract the value of the key 'total' from achievements dict
def extract_achievements_total(val):
    """
    Extract total number of achievements from Steam metadata.
    Steam stores achievements as a stringified dictionary containing
    a 'total' field.
    Returns 0 when:
    - value is NaN or missing
    - parsing fails
    - structure is not a dict
    - 'total' field is absent
    Ensures safe numeric output for ML pipelines.
    """
    if pd.isna(val): 
        return 0
    try:
        parsed = ast.literal_eval(val)
        if isinstance(parsed, dict):
            return int(parsed.get('total', 0))
        return 0
    except (ValueError, SyntaxError, TypeError):
        return 0

# Extract feature from JSON structures
def extract_json_features(df):
    """
    Flatten JSON-like Steam metadata into ML-ready features.
    Extracts:
    - Platform support (Windows/Mac/Linux as binary flags)
    - metacritic_score
    - num_achievements
    - categories, genres, tags (normalized token lists)
    - publishers, developers (normalized token lists)
    Converts nested structures into tabular features suitable for ML
    Ensures compatibility with standard machine learning pipelines
    """

    print("Extraction features from JSON structures...")
    
    if 'platforms' in df.columns:
        platforms_parsed = df['platforms'].apply(parse_dict)
        df['platform_windows'] = platforms_parsed.apply(lambda x: x.get('windows', False)).astype(int)
        df['platform_mac'] = platforms_parsed.apply(lambda x: x.get('mac', False)).astype(int)
        df['platform_linux'] = platforms_parsed.apply(lambda x: x.get('linux', False)).astype(int)

    if 'metacritic' in df.columns:
        df['metacritic_score'] = df['metacritic'].apply(lambda x: parse_dict(x).get('score', 0))

    # Extracting categorical features into lists
    if 'categories' in df.columns: df['categories'] = df['categories'].apply(extract_list_from_dicts)
    if 'genres' in df.columns: df['genres'] = df['genres'].apply(extract_list_from_dicts)
    if 'publishers' in df.columns: df['publishers'] = df['publishers'].apply(parse_simple_list)
    if 'developers' in df.columns: df['developers'] = df['developers'].apply(parse_simple_list)
    if 'tags' in df.columns: df['tags'] = df['tags'].apply(extract_keys_from_dict)
    
    if 'achievements' in df.columns:
        df['num_achievements'] = df['achievements'].apply(extract_achievements_total)
    return df


"""
Text Processing
└─ clean_text_descriptions
"""
# Cleaning HTML tags from descriptions
def clean_text_descriptions(df):
    """
    Remove HTML tags and normalize whitespace in Steam text fields.
    Cleans:
    - name_store, name
    - detailed_description
    - short_description
    Steps:
    - Strip HTML markup
    - Normalize repeated whitespace
    - Trim leading/trailing spaces
    Improves text quality for NLP tasks like TF-IDF, embeddings,
    and classification by removing formatting noise
    """

    print("Cleaning HTML tags and malformed characters from text descriptions...")
    
    for col in ['name_store', 'name', 'detailed_description', 'short_description']:
        if col in df.columns:
            # Cast to string
            df[col] = df[col].astype(str)
            # Removing HTML tags
            df[col] = df[col].str.replace(r'<[^>]+>', ' ', regex=True)
            # Removing newline, carriage return and tabs
            df[col] = df[col].str.replace(r'[\n\r\t]+', ' ', regex=True)
            # Removing multiple spaces
            df[col] = df[col].str.replace(r'\s+', ' ', regex=True).str.strip()
            
            # Marking as NaN the nan string
            df.loc[df[col] == 'nan', col] = np.nan
            
    return df

"""
Hardware Requirement Features
├─ extract_ram_features
└─ extract_gpu_cpu_features
"""
def extract_ram_features(df):
    """
    Extract and normalize RAM requirements from system specs.
    Creates:
    - min_ram_gb
    - rec_ram_gb
    Parses semi-structured requirement text and converts values
    into standardized gigabytes.
    Handles formats like MB/GB and HTML-laced strings.
    Provides a proxy for game technical complexity.
    """
    print("Extracting min and recommended ram features...")

    def get_req_string(val, req_type):
        if pd.isna(val) or val == '[]': 
            return ""
        try:
            parsed = ast.literal_eval(val)
            if isinstance(parsed, dict):
                return parsed.get(req_type, "")
        except: 
            pass
        return ""

    def parse_ram_to_gb(text):
        if not text or pd.isna(text): 
            return np.nan
            
        text = str(text).lower()
        # Preventive cleaning
        text = re.sub(r'(video\s*memory|vram|graphics\s*memory|storage|hard\s*drive|space)', '', text)
        
        ram_pattern = re.compile(r'(?:memory[:\s-]*(\d+[\.,]?\d*)\s*(mb|gb|kb))|(?:(\d+[\.,]?\d*)\s*(mb|gb|kb)\s*ram)')
        matches = ram_pattern.findall(text)
        
        if not matches: 
            return np.nan
            
        match = matches[0]
        value_str = match[0] if match[0] else match[2]
        unit = match[1] if match[1] else match[3]
        
        try:
            value = float(value_str.replace(',', '.'))
            if unit == 'kb': 
                return value / (1024.0 * 1024.0)
            elif unit == 'mb': 
                return value / 1024.0
            else: 
                return value
        except: 
            return np.nan

    # Processing minimum and recommended
    for req_type in ['minimum', 'recommended']:
        req_strings = (
            df.get('pc_requirements', pd.Series([""]*len(df))).apply(lambda x: get_req_string(x, req_type)) + " " +
            df.get('mac_requirements', pd.Series([""]*len(df))).apply(lambda x: get_req_string(x, req_type)) + " " +
            df.get('linux_requirements', pd.Series([""]*len(df))).apply(lambda x: get_req_string(x, req_type))
        )
        
        # HTML cleaning
        req_strings = req_strings.str.replace(r'<[^>]+>', ' ', regex=True)
        
        # Creating columns
        col_name = 'min_ram_gb' if req_type == 'minimum' else 'rec_ram_gb'
        df[col_name] = req_strings.apply(parse_ram_to_gb)

    df['rec_ram_gb'] = df['rec_ram_gb'].fillna(df['min_ram_gb'])
    return df

def extract_gpu_cpu_features(df):
    """
    Extract hardware requirement signals from system specs text.
    Creates binary indicators for:
    - req_high_end_gpu
    - req_dedicated_gpu
    - req_high_cpu
    - req_mid_cpu
    Uses heuristic pattern matching to estimate hardware demands.
    Not a precise benchmark, but a proxy for technical complexity.
    """
    print("Extracting CPU/GPU features (vectorized)...")
    
    reqs = (
        df.get('pc_requirements', pd.Series(['']*len(df))).fillna('') + ' ' +
        df.get('mac_requirements', pd.Series(['']*len(df))).fillna('') + ' ' +
        df.get('linux_requirements', pd.Series(['']*len(df))).fillna('')
    ).str.lower()
    
    # Boolean flags (0/1) indicating whether a specific CPU/GPU tier is required
    df['req_high_end_gpu'] = reqs.str.contains(r'rtx\s*\d{4}|rx\s*\d{4}|radeon\s*vii', regex=True).astype(int)
    df['req_dedicated_gpu'] = reqs.str.contains(r'gtx|geforce|radeon\s*r|nvidia|amd', regex=True).astype(int)
    df['req_high_cpu'] = reqs.str.contains(r'i7|i9|ryzen\s*5|ryzen\s*7', regex=True).astype(int)
    df['req_mid_cpu'] = reqs.str.contains(r'i5|ryzen\s*3|fx-', regex=True).astype(int)
    
    return df

"""
Financial and Temporal Features
└─ extract_financial_and_temporal
"""
def extract_financial_and_temporal(df):
    """
    Extract financial, temporal, and engagement features.
    Creates:
    - price
    - release_year
    - release_month
    - review_ratio
    Removes invalid or future-dated releases.
    Review ratio measures user sentiment from positive/negative reviews.
    Captures pricing, age, and popularity signals for modeling.
    """

    print("Extracting financial and time features...")

    if 'initialprice' in df.columns:
        df['price'] = df['initialprice'] / 100.0  
        
        # Check using is_free
        if 'is_free' in df.columns:
            # If is_free == True the price and discount are 0.0
            df.loc[df['is_free'] == True, 'price'] = 0.0
            df.loc[df['is_free'] == True, 'discount'] = 0.0
        else:
            df['is_free'] = (df['price'] == 0.0)

    def parse_date(date_str):
        if not date_str: return pd.NaT
        return pd.to_datetime(date_str, errors='coerce')

    if 'release_date' in df.columns:
        parsed_dates = df['release_date'].apply(lambda x: parse_dict(x).get('date', '') if isinstance(x, str) and '{' in x else str(x))
        dates = parsed_dates.apply(parse_date)
        
        # Temp column for using filters
        df['temp_date'] = dates
        initial_len = len(df)
        
        # Drop rows with a missing release date or a release date in the future (not yet released)
        reference_date = pd.to_datetime(datetime.now().date())
        df = df[df['temp_date'].notna() & (df['temp_date'] <= reference_date)].copy()
        print(f"   -> Dropped {initial_len - len(df)} rows with missing or future release dates.")

        # Extracting month and year
        df['release_year'] = df['temp_date'].dt.year.astype(int)
        df['release_month'] = df['temp_date'].dt.month.astype(int)
        
        # Dropping temp column
        df = df.drop(columns=['temp_date'])

    if 'positive' in df.columns and 'negative' in df.columns:
        total_reviews = df['positive'] + df['negative']
        df['review_ratio'] = np.where(total_reviews > 0, df['positive'] / total_reviews, 0)
    return df

"""
Language Features
└─ extract_language_features
"""
# Extracting language features
def extract_language_features(df):
    """
    Normalize Steam language data into ML features.
    Creates:
    - languages (cleaned token list)
    - num_languages_supported
    Converts comma-separated strings into structured features.
    Missing values become empty lists.
    Serves as a proxy for localization effort and market reach.
    """

    print("Extracting language features...")
    if 'languages' in df.columns:
        
        df['languages'] = df['languages'].fillna('').apply(
            lambda x: [lang.strip().replace(' ', '_') for lang in str(x).split(',') if lang.strip()]
        )
        
        df['num_languages_supported'] = df['languages'].apply(len)
    return df

"""
Advanced quality filter
├─ drops games without name or descriptions
├─ drops games with names/descriptions/tags containing CJK or Cyrillic scripts
└─ imputes/normalizes remaining NaN values in numeric and binary columns
"""
def advanced_quality_filtering(df):
    print("Applying advanced quality filters (NaN dropping, CJK/Cyrillic removal, imputations)...")
    
    initial_len = len(df)
    
    # Drop games without name or descriptions
    df = df.dropna(subset=['name_store', 'short_description', 'detailed_description']).copy()
    
    # Drops games with Chinese, Japanese, Korean, or Cyrillic characters
    cjk_cyrillic_pattern = re.compile(r'[\u4e00-\u9fff\u3040-\u30ff\uac00-\ud7af\u0400-\u04ff]')
    
    def has_foreign_chars(val):
        # If it's a list, convert it to a single string before matching
        if isinstance(val, (list, np.ndarray)):
            val = " ".join(str(x) for x in val)
    
        if pd.isna(val): 
            return False
        
        return bool(cjk_cyrillic_pattern.search(str(val)))

    # Dropping the lines
    for col in ['name_store', 'short_description', 'detailed_description', 'tags', 'genres', 'publishers', 'developers']:
        if col in df.columns:
            mask = df[col].apply(has_foreign_chars)
            df = df[~mask]
    
    # Cleaning required_age (NaN -> 0)
    if 'required_age' in df.columns:
        df['required_age'] = pd.to_numeric(df['required_age'], errors='coerce').fillna(0).astype(int)
        
    # Cleaning NaN playtime
    playtime_cols = ['average_2weeks', 'median_2weeks', 'median_forever', 'average_forever']
    for col in playtime_cols:
        if col in df.columns:
            # NaN -> 0
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

    # Sanitize binary columns: coerce to numeric, fill NaN with 0, and clip to [0, 1]
    binary_cols = ['is_controller_supported', 'has_third_party_drm', 'requires_ext_account', 
                   'platform_windows', 'platform_mac', 'platform_linux', 
                   'req_high_end_gpu', 'req_dedicated_gpu', 'req_high_cpu', 'req_mid_cpu']
    for col in binary_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
            df[col] = df[col].clip(0, 1)

    print(f"   -> Dropped {initial_len - len(df)} rows due to quality filtering (NaNs or Foreign Chars).")
    return df

"""
Dataset Organization
├─ reorder_and_rename_columns
└─ clean_and_export
"""
def reorder_and_rename_columns(df):
    """
    Standardize column names and enforce ML-ready column ordering.
    Groups features into logical categories (game metadata, content,
    language, engagement, pricing, and engineered features).
    Renames:
    - name_store -> name
    Ensures target_owners is placed last to avoid leakage.
    Improves readability, debugging, and reproducibility of the dataset.
    """

    print("Renaming and reordering columns...")
    
    df = df.rename(columns={'name_store': 'name'})
    
    group_name = ['name', 'required_age', 'developers', 'publishers', 'release_year', 'release_month']
    group_topic = ['categories', 'genres', 'tags', 'detailed_description', 'short_description', 'num_dlc', 'num_achievements']
    group_lan = ['languages', 'num_languages_supported']
    group_scores = ['metacritic_score', 'review_ratio']
    group_price = ['is_free', 'price', 'discount']
    
    new_cols = ['name'] if 'name' in df.columns else []
    
    for col in group_name + group_topic + group_lan + group_scores + group_price:
        if col in df.columns and col not in new_cols:
            new_cols.append(col)
            
    for col in df.columns:
        if col not in new_cols and col != 'target_owners':
            new_cols.append(col)
            
    if 'target_owners' in df.columns:
        new_cols.append('target_owners')
        
    return df[new_cols]

def clean_and_export(df, output_filename='steam_dataset_ready.csv'):
    """
    Final dataset cleanup before export.
    Removes:
    - Identifiers (no predictive value)
    - Raw processed columns (already engineered)
    - Duplicate merge artifacts
    - Target leakage features (e.g., ccu, positive, negative)
    - Irrelevant metadata
    Exports final ML-ready dataset to CSV and prints shape.
    Ensures clean separation between features and target signal.
    """
    print("Final cleaning...")

    cols_to_drop = [
        # Unique IDs of the games
        'steam_appid', 'appid', 'name_spy',
        
        # Features with zero variance
        'type', 'fullgame', 'score_rank', 'userscore',

        # Lists of links
        'movies', 'screenshots', 'header_image', 'background', 'website',

        # Already processed data
        'pc_requirements', 'mac_requirements', 'linux_requirements', 'reviews', 'platforms',
        'owners', 'owners_lower_bound', 'release_date', 'initialprice', 'achievements',

        # Redundant features (due to datasets merge)
        'developer', 'publisher', 'supported_languages', 'metacritic',
        'price_overview', 'about_the_game', 'packages', 'package_groups', 'demos',
        'content_descriptors', 'recommendations', 'genre', 'is_free',

        # Features too correlated to our target (number of users) 
        'ccu', 'positive', 'negative',

        # Useless feature for our domain (EULA standard, support mails, ...)
        'legal_notice', 'support_info'
    ]
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors='ignore')

    # Saving CSV
    Path(output_filename).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_filename, index=False, quoting=csv.QUOTE_MINIMAL, escapechar='\\')
    print(f"Dataset saved into '{output_filename}' with shape: {df.shape}")
    return df

In [ ]:
FILE_STEAM_STORE = "../dataset/raw_data/steam_app_data.zip"
FILE_STEAM_SPY = "../dataset/raw_data/steamspy_data.zip"

df_merged = load_and_merge(FILE_STEAM_STORE, FILE_STEAM_SPY)
df_filtered = process_game_features(df_merged)
df_target = process_target_owners(df_filtered)

# Check the data
display(df_target[['owners', 'target_owners']].head())

In [ ]:
df_json = extract_json_features(df_target)
df_lang = extract_language_features(df_json)
df_ram = extract_ram_features(df_lang)
df_hw = extract_gpu_cpu_features(df_ram)
df_fin_temp = extract_financial_and_temporal(df_hw)
df_res = extract_restrictions_features(df_fin_temp)
df_clean_text = clean_text_descriptions(df_res)
df_advanced = advanced_quality_filtering(df_clean_text)
df_reordered = reorder_and_rename_columns(df_advanced)

df_final = clean_and_export(df_reordered, output_filename="../dataset/clean_data/clean_dataset.csv")

# Grid Search (TF-IDF & PLS-DA)
Fast grid search to reduce the computation time of hyperparameter_tuning and estimate the best number of textual features (`max_tfidf_features` and `pls_da__n_components`)

In [ ]:
class CorrelationRemover(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.97):
        self.threshold = threshold

    def fit(self, X, y=None):
        corr_matrix = X.corr().abs()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        self.to_drop_ = [column for column in upper.columns if any(upper[column] > self.threshold)]
        return self

    def transform(self, X):
        return X.drop(columns=self.to_drop_, errors='ignore')

def dynamic_undersample(y):
    counts = pd.Series(y).value_counts().to_dict()
    target_class_0 = int(counts[0] * 0.25)
    safe_target = max(target_class_0, counts.get(1, 0))
    return {0: safe_target}

class DynamicSMOTENC(BaseEstimator):
    def __init__(self, sampling_strategy='auto', k_neighbors=5, random_state=None):
        self.sampling_strategy = sampling_strategy
        self.k_neighbors = k_neighbors
        self.random_state = random_state

    def fit(self, X, y=None):
        return self

    def fit_resample(self, X, y):
        is_df = isinstance(X, pd.DataFrame)
        columns = X.columns if is_df else None
        cat_mask = [X[col].nunique() <= 2 for col in X.columns] if is_df else None

        smote = SMOTENC(
            categorical_features=cat_mask,
            sampling_strategy=self.sampling_strategy,
            k_neighbors=self.k_neighbors,
            random_state=self.random_state
        )

        with sklearn.config_context(transform_output="default"):
            X_res, y_res = smote.fit_resample(X, y)

        if is_df and not isinstance(X_res, pd.DataFrame):
            X_res = pd.DataFrame(X_res, columns=columns)
        return X_res, y_res

def dynamic_oversample(y):
    counts = pd.Series(y).value_counts().to_dict()
    strategy = {}
    if 4 in counts:
        base_reference = counts.get(3, counts[4])
        target_4 = max(counts[4], int(base_reference * 0.5))
        if target_4 > counts[4]:
            strategy[4] = target_4
    return strategy

class FeatureNameSanitizer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            X_clean = X.copy()
            X_clean.columns = X_clean.columns.str.replace(r'[\[\]<]', '_', regex=True)
            return X_clean
        return X

def precompute_detailed_embeddings(df, text_col='detailed_description'):
    print("Pre-compuation of textual embeddings...")
    model = SentenceTransformer('all-mpnet-base-v2', device=DEVICE)
    texts = df[text_col].fillna("").tolist()
    embeddings = model.encode(texts, show_progress_bar=True)
    
    emb_cols = [f"raw_emb_{i}" for i in range(embeddings.shape[1])]
    df_emb = pd.DataFrame(embeddings, columns=emb_cols, index=df.index)
    
    del model
    if DEVICE == 'cuda': torch.cuda.empty_cache()
        
    return pd.concat([df.drop(columns=[text_col], errors='ignore'), df_emb], axis=1), emb_cols

def get_base_preprocessor(numeric_cols, categorical_cols):
    numeric_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='median')), 
        ('scaler', RobustScaler())
    ])
    categorical_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False)) 
    ])
    return ColumnTransformer(
        transformers=[
            ('num_pipeline', numeric_transformer, numeric_cols),
            ('cat_pipeline', categorical_transformer, categorical_cols)
        ],
        remainder='passthrough',
        verbose_feature_names_out=False
    )

class SteamFeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, top_n_creators=50, max_tfidf_features=30):
        self.top_n_creators = top_n_creators
        self.max_tfidf_features = max_tfidf_features
    
    def fit(self, X, y=None):
        self.mlb_cat_ = MultiLabelBinarizer().fit(X['categories'].tolist() if 'categories' in X.columns else [])
        self.mlb_genres_ = MultiLabelBinarizer().fit(X['genres'].tolist() if 'genres' in X.columns else [])
        self.mlb_tags_ = MultiLabelBinarizer().fit(X['tags'].tolist() if 'tags' in X.columns else [])
        self.mlb_langs_ = MultiLabelBinarizer().fit(X['languages'].tolist() if 'languages' in X.columns else [])
        
        self.top_publishers_ = X['publishers'].explode().dropna().value_counts().head(self.top_n_creators).index.tolist() if 'publishers' in X.columns else []
        self.top_developers_ = X['developers'].explode().dropna().value_counts().head(self.top_n_creators).index.tolist() if 'developers' in X.columns else []
        
        short_desc = X['short_description'].fillna("") if 'short_description' in X.columns else pd.Series([""]*len(X))
        self.tfidf_ = TfidfVectorizer(max_features=self.max_tfidf_features, stop_words='english', ngram_range=(1, 2))
        self.tfidf_.fit(short_desc)
        
        if 'min_ram_gb' in X.columns:
            self.ram_imputer_ = SimpleImputer(strategy='median')
            self.ram_imputer_.fit(X[['min_ram_gb']])
        return self
    
    def transform(self, X):
        df_out = X.copy()
        new_features = []
        for col, mlb, prefix in [('categories', self.mlb_cat_, 'cat'), ('genres', self.mlb_genres_, 'genre'),
                                 ('tags', self.mlb_tags_, 'tag'), ('languages', self.mlb_langs_, 'lang')]:
            if col in df_out.columns:
                encoded = pd.DataFrame(mlb.transform(df_out[col]), columns=[f"{prefix}_{c}" for c in mlb.classes_], index=df_out.index)
                new_features.append(encoded)

        if 'publishers' in df_out.columns:
            pub_cols = {f'pub_{str(p).replace(" ", "_")}': df_out['publishers'].apply(lambda x: 1 if p in x else 0) for p in self.top_publishers_}
            new_features.append(pd.DataFrame(pub_cols, index=df_out.index))

        if 'developers' in df_out.columns:
            dev_cols = {f'dev_{str(d).replace(" ", "_")}': df_out['developers'].apply(lambda x: 1 if d in x else 0) for d in self.top_developers_}
            new_features.append(pd.DataFrame(dev_cols, index=df_out.index))

        if 'short_description' in df_out.columns:
            tfidf_matrix = self.tfidf_.transform(df_out['short_description'].fillna(""))
            tfidf_cols = [f"tfidf_{w.replace(' ', '_')}" for w in self.tfidf_.get_feature_names_out()]
            df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_cols, index=df_out.index)
            new_features.append(df_tfidf)

        if new_features: df_out = pd.concat([df_out] + new_features, axis=1)

        if hasattr(self, 'ram_imputer_') and 'min_ram_gb' in df_out.columns:
            df_out['min_ram_gb'] = self.ram_imputer_.transform(df_out[['min_ram_gb']])
        if 'rec_ram_gb' in df_out.columns:
            df_out['rec_ram_gb'] = df_out['rec_ram_gb'].fillna(df_out['min_ram_gb'])

        cols_to_drop = ['categories', 'genres', 'tags', 'publishers', 'developers', 'languages', 'short_description']
        df_out = df_out.drop(columns=[c for c in cols_to_drop if c in df_out.columns], errors='ignore')
        return df_out.select_dtypes(include=['number', 'bool'])

class SupervisedPLSDATransformer(BaseEstimator, TransformerMixin):
    def __init__(self, emb_cols, n_components=50):
        self.emb_cols = emb_cols
        self.n_components = n_components

    def fit(self, X, y=None):
        if y is None: raise ValueError("PLS-DA needs target y for training")
        self.pls = PLSRegression(n_components=self.n_components)
        self.ohe = OneHotEncoder(sparse_output=False)
        self._transformed_cols = [col for col in self.emb_cols if col in X.columns]
        if not self._transformed_cols: raise ValueError("No embeddings column found for PLS-DA.")
        
        X_target = X[self._transformed_cols]
        y_encoded = self.ohe.fit_transform(np.array(y).reshape(-1, 1))
        self.pls.fit(X_target, y_encoded)
        return self

    def transform(self, X):
        X_target = X[self._transformed_cols]
        X_remainder = X.drop(columns=self._transformed_cols)
        X_trans = self.pls.transform(X_target)
        if isinstance(X_trans, pd.DataFrame): X_trans = X_trans.to_numpy()
        pls_col_names = [f"pls_da_{i}" for i in range(self.n_components)]
        df_pls = pd.DataFrame(X_trans, columns=pls_col_names, index=X.index)
        return pd.concat([X_remainder, df_pls], axis=1)

class WeightedXGBClassifier(XGBClassifier):
    def fit(self, X, y, **kwargs):
        weights = compute_sample_weight(class_weight='balanced', y=y)
        return super().fit(X, y, sample_weight=weights, **kwargs)

In [ ]:
# Configuration (Global SEED and PRE_RELEASE moved to the top)
SAMPLE_FRAC = 1.0
CV_FOLDS = 3

DATASET_PATH = "../dataset/clean_data/clean_dataset.csv"
df_grid = pd.read_csv(DATASET_PATH)

if PRE_RELEASE:
    post_release_feature = [
        'num_achievements', 'metacritic_score', 'review_ratio', 'num_dlc',
        'discount', 'average_forever', 'average_2weeks', 'median_forever', 'median_2weeks'
    ]
    df_grid = df_grid.drop(columns=[c for c in post_release_feature if c in df_grid.columns])

X = df_grid.drop(columns=['target_owners', 'name'])
y = df_grid['target_owners'].astype(int)

if SAMPLE_FRAC < 1.0:
    X, _, y, _ = train_test_split(X, y, train_size=SAMPLE_FRAC, stratify=y, random_state=SEED)

list_columns = ['categories', 'genres', 'tags', 'publishers', 'developers', 'languages']
for col in list_columns:
    if col in X.columns:
        X[col] = X[col].apply(lambda v: ast.literal_eval(v) if isinstance(v, str) else v)

df_temp = X.copy()
df_temp['target_owners'] = y
df_processed, emb_cols = precompute_detailed_embeddings(df_temp, text_col='detailed_description')

X_proc = df_processed.drop(columns=['target_owners'])
y_proc = df_processed['target_owners']

all_numeric_cols = [
    'price', 'release_year', 'min_ram_gb', 'rec_ram_gb', 'required_age',
    'num_dlc', 'num_achievements', 'num_languages_supported', 'metacritic_score',
    'review_ratio', 'discount', 'average_forever', 'average_2weeks',
    'median_forever', 'median_2weeks'
]
numeric_cols = [c for c in all_numeric_cols if c in X_proc.columns]
categorical_cols = ['release_month']
base_preprocessor = get_base_preprocessor(numeric_cols, categorical_cols)

probe_models = {
    'DecisionTree': DecisionTreeClassifier(class_weight='balanced', random_state=SEED),
    'RandomForest': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=SEED, n_jobs=-1),
    'XGBoost': WeightedXGBClassifier(n_estimators=100, random_state=SEED, n_jobs=-1,
                                    eval_metric='mlogloss', tree_method='hist', device=DEVICE)
}

tfidf_grid = [15, 30, 50]
pls_grid = [20, 50, 80]

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)

results = []
total = len(probe_models) * len(tfidf_grid) * len(pls_grid)
done = 0

for model_name, clf in probe_models.items():
    for n_tfidf in tfidf_grid:
        for n_pls in pls_grid:
            done += 1
            print(f"[{done}/{total}] {model_name} | tfidf={n_tfidf} | pls={n_pls}")

            pipe = ImbPipeline([
                ('steam_extractor', SteamFeatureExtractor(max_tfidf_features=n_tfidf)),
                ('base_preprocessor', base_preprocessor),
                ('rus', RandomUnderSampler(sampling_strategy=dynamic_undersample, random_state=SEED)),
                ('pls_da', SupervisedPLSDATransformer(emb_cols=emb_cols, n_components=n_pls)),
                ('corr_remover', CorrelationRemover(threshold=0.95)),
                ('smote_nc', DynamicSMOTENC(sampling_strategy=dynamic_oversample, k_neighbors=3, random_state=SEED)),
                ('tomek', TomekLinks()),
                ('sanitizer', FeatureNameSanitizer()),
                ('classifier', clf)
            ])

            try:
                scores = cross_val_score(pipe, X_proc, y_proc, cv=cv, scoring=qwk_scorer,
                                          n_jobs=1, error_score='raise')
                mean_score, std_score = scores.mean(), scores.std()
            except Exception as e:
                print(f"   -> FAILED: {e}")
                mean_score, std_score = np.nan, np.nan

            results.append({
                'model': model_name,
                'max_tfidf_features': n_tfidf,
                'pls_da_n_components': n_pls,
                'qwk_mean': mean_score,
                'qwk_std': std_score
            })

results_df = pd.DataFrame(results)
results_df.to_csv("tfidf_pls_grid_results.csv", index=False)

print("\n" + "="*60)
print(" BEST COMBINATIONS")
print("="*60)
best_per_model = results_df.loc[results_df.groupby('model')['qwk_mean'].idxmax()]
print(best_per_model.to_string(index=False))

print("\n" + "="*60)
print(" MEAN VALUE BETWEEN THE MODELS")
print("="*60)
avg_across_models = results_df.groupby(['max_tfidf_features', 'pls_da_n_components'])['qwk_mean'].mean()
avg_across_models = avg_across_models.reset_index().sort_values('qwk_mean', ascending=False)
print(avg_across_models.to_string(index=False))

In [ ]:
class NumpyEncoder(json.JSONEncoder):
    """
    Custom JSON encoder to seamlessly convert NumPy data types 
    into native Python types for JSON serialization
    """
    def default(self, obj):
        if isinstance(obj, np.integer): return int(obj)
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        if isinstance(obj, np.bool_): return bool(obj)
        return super(NumpyEncoder, self).default(obj)

In [ ]:
# Models configuration
models_config = {
    'DecisionTree': {
        'selector_estimator': DecisionTreeClassifier(class_weight='balanced', random_state=SEED),
        'estimator': DecisionTreeClassifier(class_weight='balanced', random_state=SEED),
        'n_iter': 5,
        'tfidf_features': 15 if PRE_RELEASE else 50,
        'pls_components': 20,
        'param_distributions': {
            'classifier__max_depth': IntDistribution(5, 30),
            'classifier__min_samples_split': IntDistribution(2, 20),
            'classifier__min_samples_leaf': IntDistribution(1, 10),
            'feature_selection__threshold': CategoricalDistribution(['median', 'mean'])
        }
    },
    'RandomForest': {
        'selector_estimator': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=SEED, n_jobs=-1),
        'estimator': RandomForestClassifier(class_weight='balanced', random_state=SEED, n_jobs=-1),
        'n_iter': 40,
        'tfidf_features': 15 if PRE_RELEASE else 30,
        'pls_components': 20,
        'param_distributions': {
            'classifier__n_estimators': IntDistribution(100, 400),
            'classifier__max_depth': IntDistribution(5, 30),
            'classifier__min_samples_split': IntDistribution(2, 15),
            'classifier__min_samples_leaf': IntDistribution(1, 8),
            'classifier__max_features': FloatDistribution(0.1, 0.6),
            'feature_selection__threshold': CategoricalDistribution(['median', 'mean'])
        }
    },
    'XGBoost': {
        'selector_estimator': WeightedXGBClassifier(n_estimators=100, random_state=SEED, n_jobs=-1, eval_metric='mlogloss', tree_method='hist', device=DEVICE),
        'estimator': WeightedXGBClassifier(random_state=SEED, n_jobs=-1, eval_metric='mlogloss', tree_method='hist', device=DEVICE),
        'n_iter': 50,
        'tfidf_features': 15,
        'pls_components': 20,
        'param_distributions': {
            'classifier__n_estimators': IntDistribution(100, 400),
            'classifier__max_depth': IntDistribution(3, 9),
            'classifier__learning_rate': FloatDistribution(0.05, 0.19),
            'classifier__min_child_weight': IntDistribution(1, 8),
            'classifier__subsample': FloatDistribution(0.6, 1.0),
            'classifier__reg_lambda': FloatDistribution(0.5, 3.0),
            'feature_selection__threshold': CategoricalDistribution(['median', 'mean'])
        }
    }
}

# Setup Cross Validation Nested
cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
all_results = {}

for model_name, config in models_config.items():
    print(f"\n{'='*40}")
    print(f"Satrting evaluation Nested CV: {model_name}")
    print(f"{'='*40}")

    cache_dir = os.path.join(".", "ml_cache", f"pipeline_cache_{model_name}")
    memory = Memory(location=cache_dir, verbose=0)
    
    # Costruzione della pipeline Imblearn riutilizzando le classi già istanziate
    base_pipe = ImbPipeline([
        ('steam_extractor', SteamFeatureExtractor(max_tfidf_features=config['tfidf_features'])),
        ('base_preprocessor', base_preprocessor),
        ('rus', RandomUnderSampler(sampling_strategy=dynamic_undersample, random_state=SEED)),
        ('pls_da', SupervisedPLSDATransformer(emb_cols=emb_cols, n_components=config['pls_components'])),
        ('corr_remover', CorrelationRemover(threshold=0.95)),
        ('smote_nc', DynamicSMOTENC(sampling_strategy=dynamic_oversample, k_neighbors=3, random_state=SEED)),
        ('tomek', TomekLinks()),
        ('sanitizer', FeatureNameSanitizer()),
        ('feature_selection', SelectFromModel(config['selector_estimator'])),
        ('classifier', config['estimator'])
    ], memory=memory)

    model_results = {
        'folds_data': [],
        'best_params_final_fit': None,
        'final_selected_features': None
    }

    outer_fold_idx = 1
    for train_ix, test_ix in tqdm(cv_outer.split(X_proc, y_proc), total=cv_outer.n_splits, desc=f"Outer CV ({model_name})"):
        X_train, X_test = X_proc.iloc[train_ix], X_proc.iloc[test_ix]
        y_train, y_test = y_proc.iloc[train_ix], y_proc.iloc[test_ix]

        # Inner CV handled by OptunaSearchCV
        search = OptunaSearchCV(
            estimator=base_pipe,
            param_distributions=config['param_distributions'],
            n_trials=config['n_iter'],
            cv=cv_inner,
            scoring=qwk_scorer,
            random_state=SEED,
            n_jobs=1, 
            verbose=0,
            error_score='raise'
        )

        search.fit(X_train, y_train)
        best_pipe_fold = search.best_estimator_
        selected_features = list(best_pipe_fold.named_steps['feature_selection'].get_feature_names_out())

        y_pred = best_pipe_fold.predict(X_test)
        
        model_results['folds_data'].append({
            'fold': outer_fold_idx,
            'best_params_fold': search.best_params_,
            'selected_features_count': len(selected_features),
            'selected_features': selected_features,
            'metrics': {
                'accuracy': accuracy_score(y_test, y_pred),
                'precision_macro': precision_score(y_test, y_pred, average='macro', zero_division=0),
                'recall_macro': recall_score(y_test, y_pred, average='macro', zero_division=0),
                'f1_macro': f1_score(y_test, y_pred, average='macro'),
                'quadratic_weighted_kappa': cohen_kappa_score(y_test, y_pred, weights='quadratic')
            },
            'confusion_matrix': confusion_matrix(y_test, y_pred).tolist()
        })
        outer_fold_idx += 1
        
        memory.clear(warn=False)
        if os.path.exists(cache_dir):
            shutil.rmtree(cache_dir)

    # Final fit per estrarre i parametri ottimali sull'intero dataset elaborato
    print(f"Extracting optimal hyperparameters for {model_name}...")
    final_search = OptunaSearchCV(
        estimator=base_pipe, 
        param_distributions=config['param_distributions'], 
        n_trials=config['n_iter'],
        cv=cv_inner, 
        scoring=qwk_scorer, 
        random_state=SEED,
        n_jobs=1, 
        error_score='raise'
    )
    final_search.fit(X_proc, y_proc)
    
    model_results['best_params_final_fit'] = final_search.best_params_
    final_selected = list(final_search.best_estimator_.named_steps['feature_selection'].get_feature_names_out())
    model_results['final_selected_features_count'] = len(final_selected)
    model_results['final_selected_features'] = final_selected
    
    all_results[model_name] = model_results

    memory.clear(warn=False)
    if os.path.exists(cache_dir):
        shutil.rmtree(cache_dir)

# Export dei risultati
output_file = f"tuning_results_{'pre_release' if PRE_RELEASE else 'post_release'}.json"
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(all_results, f, indent=4, cls=NumpyEncoder)

print(f"\nTuning completed. Results saved in '{output_file}'")

In [ ]:
def load_data(json_path):
    """Carica i risultati del tuning nested-CV."""
    with open(json_path, 'r', encoding='utf-8') as f:
        return json.load(f)

def extract_metrics_to_dataframe(results):
    """Appiattisce le metriche in un DataFrame pronto per l'analisi statistica."""
    records = []
    for model_name, model_data in results.items():
        for fold_data in model_data.get('folds_data', []):
            metrics = fold_data.get('metrics', {})
            records.append({
                'Model': model_name,
                'Fold': fold_data.get('fold'),
                'Accuracy': metrics.get('accuracy'),
                'Precision (Macro)': metrics.get('precision_macro'),
                'Recall (Macro)': metrics.get('recall_macro'),
                'F1 (Macro)': metrics.get('f1_macro'),
                'QWK': metrics.get('quadratic_weighted_kappa')
            })
    return pd.DataFrame(records)

def plot_metrics(df):
    """Disegna un boxplot per ogni metrica confrontando tutti i modelli."""
    metrics = ['QWK', 'Accuracy', 'Precision (Macro)', 'Recall (Macro)', 'F1 (Macro)']

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('Distribution of Metrics (Nested CV outer folds)', fontsize=16)
    axes = axes.flatten()

    for i, metric in enumerate(metrics):
        sns.boxplot(data=df, x='Model', y=metric, ax=axes[i], hue='Model', palette='Set2', legend=False)
        sns.stripplot(data=df, x='Model', y=metric, color='black', alpha=0.6, ax=axes[i], jitter=False)
        axes[i].set_title(metric)
        axes[i].set_xlabel('')
        axes[i].set_ylabel('Score')

    axes[-1].axis('off')
    plt.tight_layout()
    plt.show()

def perform_statistical_analysis(df, results, metric='QWK'):
    """Esegue il test di Friedman per determinare differenze statistiche globali, seguito da Wilcoxon."""
    print("\n" + "="*50)
    print(f" STATISTICAL ANALYSIS ({metric})")
    print("="*50)
    
    mean_scores = df.groupby('Model')[metric].mean().sort_values(ascending=False)
    print(f"\nAverage {metric} per model:")
    print(mean_scores.to_string())
    
    models = mean_scores.index.tolist()
    model_arrays = {model: df[df['Model'] == model][metric].values for model in models}
    
    stat, p_value = friedmanchisquare(*[model_arrays[m] for m in models])
    print(f"\nGlobal Friedman test - p-value: {p_value:.4f}")
    
    if p_value < 0.05:
        print("There is a statistically significant difference between the models")
        print("Running a Wilcoxon test between the first and the others...")
        
        best_model = models[0]
        for other_model in models[1:]:
            stat_w, p_w = wilcoxon(model_arrays[best_model], model_arrays[other_model])
            print(f" - {best_model} vs {other_model}: p-value = {p_w:.4f}")
    else:
        print("There is no statistically significant difference between the models (small sample, n=5)")
    
    best_model_name = models[0]
    print("\n" + "="*50)
    print(f" BEST MODEL: {best_model_name}")
    print("="*50)
    
    best_params = results[best_model_name].get('best_params_final_fit', {})
    final_features_count = results[best_model_name].get('final_selected_features_count', 'N/A')
    
    print(f"\nBest Hyperparameters:")
    print(json.dumps(best_params, indent=4))
    print(f"\nNumber of final features selected: {final_features_count}")

In [ ]:
# Set the JSON path
json_path = '../results/pre_release_model/tuning_results_pre_release.json'

try:
    results = load_data(json_path)
    df_metrics = extract_metrics_to_dataframe(results)
    
    # Boxplots
    plot_metrics(df_metrics)
    
    # Statistical test
    perform_statistical_analysis(df_metrics, results, metric='QWK')
    
except FileNotFoundError:
    print(f"Error: i can't find the file '{json_path}'")
except json.JSONDecodeError:
    print(f"Error: '{json_path}' isn't a valid JSON")

In [ ]:
# Configurazione del Notebook
print(f"Random Seed is set to: {SEED}")

if PRE_RELEASE:
    base_dir = "../results/pre_release_model"
    print("Mode: pre_release active. Post-release features will be ignored")
else:
    base_dir = "../results/post_release_model"
    print("Mode: post-release active. All features will be used")
    
xai_dir = os.path.join(base_dir, "xai_plots")
os.makedirs(xai_dir, exist_ok=True)
print(f"XAI plots will be saved in: {xai_dir}")

def loss_function(weights, y_proba_oof, y_train_proc):
    """Obiettivo minimizzato dalla Differential Evolution per calibrare i pesi"""
    weighted_proba = y_proba_oof * weights
    pred = np.argmax(weighted_proba, axis=1)
    return -cohen_kappa_score(y_train_proc, pred, weights='quadratic')

In [ ]:
# Caricamento e pre-processamento del dataset
DATASET_PATH = "../dataset/clean_data/clean_dataset.csv"
df_final_train = pd.read_csv(DATASET_PATH)

if PRE_RELEASE:
    post_release_feature = [
        'num_achievements', 'metacritic_score', 'review_ratio', 'num_dlc',
        'discount', 'average_forever', 'average_2weeks', 'median_forever', 'median_2weeks'
    ]
    cols_to_drop = [col for col in post_release_feature if col in df_final_train.columns]
    df_final_train = df_final_train.drop(columns=cols_to_drop)
    print(f"Post-release columns dropped: {len(cols_to_drop)} feature ignored")

X_final = df_final_train.drop(columns=['target_owners', 'name'])
y_final = df_final_train['target_owners'].astype(int)

# Parsing delle liste
list_columns = ['categories', 'genres', 'tags', 'publishers', 'developers', 'languages']
for col in list_columns:
    if col in X_final.columns:
        X_final[col] = X_final[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.20, stratify=y_final, random_state=SEED)

# Pre-computazione degli embedding per evitare ricalcoli costosi nei fold CV
df_temp_train = X_train.copy()
df_temp_train['target_owners'] = y_train
df_train_proc, emb_cols_final = precompute_detailed_embeddings(df_temp_train, text_col='detailed_description')

df_temp_test = X_test.copy()
df_temp_test['target_owners'] = y_test
df_test_proc, _ = precompute_detailed_embeddings(df_temp_test, text_col='detailed_description')

X_train_proc = df_train_proc.drop(columns=['target_owners'])
y_train_proc = df_train_proc['target_owners']
X_test_proc = df_test_proc.drop(columns=['target_owners'])
y_test_proc = df_test_proc['target_owners']

In [ ]:
# Configurazione della Pipeline Sbilanciata
all_numeric_cols = [
    'price', 'release_year', 'min_ram_gb', 'rec_ram_gb', 'required_age', 
    'num_dlc', 'num_achievements', 'num_languages_supported', 'metacritic_score', 
    'review_ratio', 'discount', 'average_forever', 'average_2weeks', 
    'median_forever', 'median_2weeks'
]
numeric_cols = [col for col in all_numeric_cols if col in X_train_proc.columns]
categorical_cols = ['release_month']

base_preprocessor_final = get_base_preprocessor(numeric_cols, categorical_cols)
plsda_step_final = SupervisedPLSDATransformer(emb_cols=emb_cols_final, n_components=20)

if PRE_RELEASE:
    print("Using pre-release hyperparameter configuration")
    xgb_estimator = WeightedXGBClassifier(
        n_estimators=397, max_depth=5, learning_rate=0.1074709860640873,
        min_child_weight=5, subsample=0.9652105402477317, reg_lambda=0.8250421259522612,
        random_state=SEED, n_jobs=-1, eval_metric='mlogloss', tree_method='hist', device=DEVICE
    )
else:
    print("Using post-release hyperparameter configuration")
    xgb_estimator = WeightedXGBClassifier(
        n_estimators=236, max_depth=7, learning_rate=0.14685290784404992,
        min_child_weight=2, subsample=0.9675284682571256, reg_lambda=0.7907934154092928,
        random_state=SEED, n_jobs=-1, eval_metric='mlogloss', tree_method='hist', device=DEVICE
    )

xgb_selector = WeightedXGBClassifier(
    n_estimators=100, random_state=SEED, n_jobs=-1, eval_metric='mlogloss', tree_method='hist', device=DEVICE
)

final_pipe = ImbPipeline([
    ('steam_extractor', SteamFeatureExtractor(max_tfidf_features=15)),
    ('base_preprocessor', base_preprocessor_final),
    ('rus', RandomUnderSampler(sampling_strategy=dynamic_undersample, random_state=SEED)),
    ('pls_da', plsda_step_final),
    ('corr_remover', CorrelationRemover(threshold=0.95)),
    ('smote_nc', DynamicSMOTENC(sampling_strategy=dynamic_oversample, k_neighbors=3, random_state=SEED)),
    ('tomek', TomekLinks()),
    ('sanitizer', FeatureNameSanitizer()),
    ('feature_selection', SelectFromModel(xgb_selector, threshold='median')),
    ('classifier', xgb_estimator)
])

print("\nFinal model training in progress...")
final_pipe.fit(X_train_proc, y_train_proc)

In [ ]:
# Calibrazione dei pesi e Valutazione Modello
print("\nWeights calibration (out-of-fold on train)...")

cv_calibration = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

y_proba_oof = cross_val_predict(
    final_pipe, X_train_proc, y_train_proc,
    cv=cv_calibration, method='predict_proba', n_jobs=1, verbose=3
)

bounds = [(0.1, 2.0)] * 5

print("\nStarting Differential Evolution to maximize Quadratic Weighted Kappa...")
result = differential_evolution(
    loss_function, bounds, args=(y_proba_oof, y_train_proc),
    strategy='best1bin', maxiter=100, popsize=15, tol=1e-3, seed=SEED, workers=-1
)

best_weights = result.x
best_weights /= np.sum(best_weights) 
print(f"-> Best weights found: {np.round(best_weights, 3)}")

# Valutazione
y_proba_test = final_pipe.predict_proba(X_test_proc)
final_proba = y_proba_test * best_weights
y_pred_adjusted = np.argmax(final_proba, axis=1)

print("\nClassification report:")
print(classification_report(y_test_proc, y_pred_adjusted))

cm = confusion_matrix(y_test_proc, y_pred_adjusted)
tier_labels = ["Tier 0", "Tier 1", "Tier 2", "Tier 3", "Tier 4"]
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=tier_labels)

fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(cmap=plt.cm.Blues, values_format='d', ax=ax)
plt.title("Confusion Matrix (calibrated thresholds)", fontsize=14, pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# Explainable AI (XAI) - Preparazione e SHAP
def _unscale_numeric_features(df_scaled, pipeline, numeric_cols):
    df_unscaled = df_scaled.copy()
    base_prep = pipeline.named_steps['base_preprocessor']
    scaler = base_prep.named_transformers_['num_pipeline'].named_steps['scaler']
    
    num_buffer = pd.DataFrame(0.0, index=df_scaled.index, columns=numeric_cols)
    for col in numeric_cols:
        if col in df_scaled.columns:
            num_buffer[col] = df_scaled[col]

    unscaled_array = scaler.inverse_transform(num_buffer)
    df_inv = pd.DataFrame(unscaled_array, columns=numeric_cols, index=df_scaled.index)
    
    for col in numeric_cols:
        if col in df_scaled.columns:
            df_unscaled[col] = df_inv[col]
    return df_unscaled

print("\nData preparation for eXplainable AI (XAI)...")
preprocessing_pipe = final_pipe[:-1]
X_test_transformed = preprocessing_pipe.transform(X_test_proc)

model = final_pipe.named_steps['classifier']
feature_names = final_pipe.named_steps['feature_selection'].get_feature_names_out()
X_test_transformed_df = pd.DataFrame(X_test_transformed, columns=feature_names)

print("Generating XGBoost Native Importance...")
plt.figure(figsize=(10, 8))
xgb.plot_importance(model, max_num_features=20, importance_type='gain', title='XGBoost Feature Importance (Gain)')
plt.tight_layout()
plt.show()

print("Calculating SHAP values...")
explainer = shap.TreeExplainer(model)
max_samples = min(2000, len(X_test_transformed_df))

if max_samples < len(X_test_transformed_df):
    _, X_test_sample, _, _ = train_test_split(
        X_test_transformed_df, y_test_proc, test_size=max_samples, stratify=y_test_proc, random_state=SEED
    )
else:
    X_test_sample = X_test_transformed_df

shap_values = explainer(X_test_sample)
X_test_sample_unscaled = _unscale_numeric_features(X_test_sample, final_pipe, numeric_cols)
shap_values.data = X_test_sample_unscaled.values

tier_labels_shap = ["Tier 0 (<20k)", "Tier 1 (20k-100k)", "Tier 2 (100k-500k)", "Tier 3 (500k-2M)", "Tier 4 (>2M)"]

for class_idx in range(5):
    print(f"--> Elaborating {tier_labels_shap[class_idx]}...")
    plt.figure(figsize=(12, 8))
    shap.plots.beeswarm(shap_values[:, :, class_idx], show=False)
    plt.title(f"Importance of the Feature for - {tier_labels_shap[class_idx]}", fontsize=14, pad=15)
    plt.tight_layout()
    plt.show()

In [ ]:
# Explainable AI (XAI) - LIME e Retraining finale
print("Executing LIME...")

X_tr_step = final_pipe.named_steps['steam_extractor'].transform(X_train_proc)
X_tr_step = final_pipe.named_steps['base_preprocessor'].transform(X_tr_step)
X_resampled, y_resampled = final_pipe.named_steps['rus'].fit_resample(X_tr_step, y_train_proc)
X_resampled = final_pipe.named_steps['pls_da'].transform(X_resampled)
X_resampled = final_pipe.named_steps['corr_remover'].transform(X_resampled)
X_resampled, y_resampled = final_pipe.named_steps['smote_nc'].fit_resample(X_resampled, y_resampled)
X_resampled, y_resampled = final_pipe.named_steps['tomek'].fit_resample(X_resampled, y_resampled)
X_resampled = final_pipe.named_steps['sanitizer'].transform(X_resampled)
X_train_transformed = final_pipe.named_steps['feature_selection'].transform(X_resampled)

X_train_transformed_df = pd.DataFrame(X_train_transformed, columns=feature_names)
X_train_unscaled = _unscale_numeric_features(X_train_transformed_df, final_pipe, numeric_cols)
lime_bg_data = X_train_unscaled.values

lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=lime_bg_data,
    feature_names=feature_names,
    class_names=['Tier 0 (<20k)', 'Tier 1 (20-100k)', 'Tier 2 (100-500k)', 'Tier 3 (500k-2M)', 'Tier 4 (>2M)'],
    mode='classification',
    random_state=SEED
)

def predict_fn_lime(x):
    df_perturbed = pd.DataFrame(x, columns=feature_names)
    base_prep = final_pipe.named_steps['base_preprocessor']
    scaler = base_prep.named_transformers_['num_pipeline'].named_steps['scaler']
    
    num_buffer = pd.DataFrame(0.0, index=df_perturbed.index, columns=numeric_cols)
    for col in numeric_cols:
        if col in df_perturbed.columns:
            num_buffer[col] = df_perturbed[col]
            
    scaled_array = scaler.transform(num_buffer)
    df_scaled_inv = pd.DataFrame(scaled_array, columns=numeric_cols, index=df_perturbed.index)
    
    df_perturbed_scaled = df_perturbed.copy()
    for col in numeric_cols:
        if col in df_perturbed.columns:
            df_perturbed_scaled[col] = df_scaled_inv[col]
            
    return model.predict_proba(df_perturbed_scaled)

for class_idx in range(5):
    candidate_positions = np.where(y_pred_adjusted == class_idx)[0]
    if len(candidate_positions) == 0: continue
    
    instance_pos = candidate_positions[0]
    instance_scaled_df = X_test_transformed_df.iloc[[instance_pos]]
    instance_unscaled_df = _unscale_numeric_features(instance_scaled_df, final_pipe, numeric_cols)
    instance_to_explain = instance_unscaled_df.iloc[0].values

    exp = lime_explainer.explain_instance(
        data_row=instance_to_explain, predict_fn=predict_fn_lime, num_features=10, labels=[class_idx]
    )
    
    fig = exp.as_pyplot_figure(label=class_idx)
    fig.tight_layout()
    plt.show()

# Final Training 100%
print("\nPreparation of the complete dataset (100%) for the production model...")
df_full = X_final.copy()
df_full['target_owners'] = y_final
df_full_proc, _ = precompute_detailed_embeddings(df_full, text_col='detailed_description')

X_full_proc = df_full_proc.drop(columns=['target_owners'])
y_full_proc = df_full_proc['target_owners']

print("Training the final model on the entire dataset...")
final_pipe.fit(X_full_proc, y_full_proc)

production_artifact = {
    'pipeline': final_pipe,
    'best_weights': best_weights
}

model_filename = 'pre_release_model.pkl' if PRE_RELEASE else 'post_release_model.pkl'
model_path = os.path.join(base_dir, model_filename)
joblib.dump(production_artifact, model_path)
print(f"Model saved in: {model_path}")